# Cubo OLAP con PySpark a partir de un modelo en estrella

Este notebook carga el modelo en estrella exportado desde SQL Server a CSV, construye un cubo OLAP con PySpark y genera visualizaciones para explicar cada operación analítica.

## 1. Preparación del entorno

Instalamos o importamos las dependencias necesarias para PySpark y las visualizaciones.

import sys
try:
    import pyspark
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyspark', 'matplotlib', 'seaborn'])
    import pyspark
    import matplotlib.pyplot as plt
    import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, expr, sum as spark_sum, when, round

sns.set(style='whitegrid', palette='muted')

## 2. Crear sesión Spark y cargar las dimensiones y la tabla de hechos

Leemos los archivos CSV en DataFrames de Spark y definimos el esquema del modelo en estrella.

spark = SparkSession.builder.appName('CuboOLAP').master('local[*]').getOrCreate()
base_path = '/workspaces/CUBO-OLAP'

dim_candidate = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/dim_candidate.csv')

dim_education = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .option('quote', 
)
    .csv(f'{base_path}/dim_education.csv')

dim_skills = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/dim_skills.csv')

dim_time = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/dim_time.csv')

fact_employability = spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{base_path}/fact_employability.csv')

print('Dimensión Candidate:')
dim_candidate.printSchema()
dim_candidate.show(5, truncate=False)

print('Dimensión Education:')
dim_education.printSchema()
dim_education.show(5, truncate=False)

print('Dimensión Skills:')
dim_skills.printSchema()
dim_skills.show(5, truncate=False)

print('Dimensión Time:')
dim_time.printSchema()
dim_time.show(5, truncate=False)

print('Tabla de Hechos Employability:')
fact_employability.printSchema()
fact_employability.show(5, truncate=False)

## 3. Construcción del cubo OLAP

Unimos la tabla de hechos con las dimensiones para obtener un cubo de análisis enriquecido.

cube_df = fact_employability
    .join(dim_candidate, 'candidate_id', 'left')
    .join(dim_education, 'education_id', 'left')
    .join(dim_skills, 'skills_id', 'left')
    .join(dim_time, 'time_id', 'left')

print('Vista del cubo OLAP:')
cube_df.printSchema()
cube_df.show(10, truncate=False)

## 4. Operaciones analíticas y visualizaciones

A continuación se muestran las consultas OLAP típicas: agregaciones, cortes, fragmentaciones y pivotes.

### 4.1 Distribución de candidatos por género y país de origen

Este gráfico muestra cómo se distribuyen los candidatos por género y por su país de origen.

gender_country = cube_df.groupBy('gender', 'country_of_origin').agg(count('*').alias('count')).orderBy('gender', 'country_of_origin')
pandas_gender_country = gender_country.toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pandas_gender_country, x='country_of_origin', y='count', hue='gender', ax=ax)
ax.set_title('Distribución de candidatos por género y país de origen')
ax.set_xlabel('País de origen')
ax.set_ylabel('Número de candidatos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.2 Nivel educativo vs. empleo

Analizamos si los candidatos con diferentes niveles de educación están empleados y la proporción de empleo por nivel.

education_employment = cube_df.groupBy('education_level').agg(
    count('*').alias('total'),
    spark_sum(when(col('employment_status') == 'Employed', 1).otherwise(0)).alias('employed')
)

education_employment = education_employment.withColumn('employment_rate', round(col('employed') / col('total') * 100, 2))
pandas_education_employment = education_employment.toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pandas_education_employment, x='education_level', y='employment_rate', palette='Blues_d', ax=ax)
ax.set_title('Tasa de empleo por nivel educativo')
ax.set_xlabel('Nivel educativo')
ax.set_ylabel('Tasa de empleo (%)')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

### 4.3 Salario promedio por sector y país de origen

Este gráfico permite ver el salario promedio por sector laboral y país de origen, un ejemplo de análisis multidimensional.

salary_sector = cube_df.groupBy('job_sector', 'country_of_origin').agg(round(avg('salary'), 2).alias('avg_salary')).orderBy('job_sector', 'country_of_origin')
pandas_salary_sector = salary_sector.toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=pandas_salary_sector, x='job_sector', y='avg_salary', hue='country_of_origin', ax=ax)
ax.set_title('Salario promedio por sector y país de origen')
ax.set_xlabel('Sector laboral')
ax.set_ylabel('Salario promedio')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 4.4 Candidatos por años desde graduación y experiencia de prácticas

Observamos la relación entre la antigüedad de graduación y el hecho de haber realizado prácticas.

experience_grad = cube_df.groupBy('years_since_graduation', 'internship_experience').agg(count('*').alias('count')).orderBy('years_since_graduation')
pandas_experience_grad = experience_grad.toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pandas_experience_grad, x='years_since_graduation', y='count', hue='internship_experience', ax=ax)
ax.set_title('Número de candidatos por años desde graduación y experiencia de prácticas')
ax.set_xlabel('Años desde graduación')
ax.set_ylabel('Cantidad de candidatos')
plt.tight_layout()
plt.show()

### 4.5 Salario promedio por rango de ranking universitario

Este gráfico muestra cómo el ranking de universidad se asocia con la compensación salarial.

salary_ranking = cube_df.groupBy('university_ranking').agg(round(avg('salary'), 2).alias('avg_salary')).orderBy('university_ranking')
pandas_salary_ranking = salary_ranking.toPandas()

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=pandas_salary_ranking, x='university_ranking', y='avg_salary', palette='viridis', ax=ax)
ax.set_title('Salario promedio por ranking universitario')
ax.set_xlabel('Ranking de la universidad')
ax.set_ylabel('Salario promedio')
plt.tight_layout()
plt.show()

## 5. Conclusiones

- El modelo en estrella usa `fact_employability` como la tabla de hechos y cuatro dimensiones (`dim_candidate`, `dim_education`, `dim_skills`, `dim_time`).
- El cubo OLAP permite explorar métricas como el número de candidatos, tasa de empleo y salario promedio a través de múltiples dimensiones.
- Las visualizaciones ilustran claramente los patrones de empleo por nivel educativo, país, sector laboral y experiencia de prácticas.